In [5]:
import torch
import torch.nn.functional as F
from torch_geometric.transforms import ToUndirected
from torch_geometric.utils import add_self_loops
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.model_selection import train_test_split

import os
import sys
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()
sys.path.append(os.path.dirname(base_dir))

from utils.graph_utils import load_graph, save_graph
from utils.sample_scoring import process_and_save, do_radical_search, do_biological_logfc, do_std
from data_processing.network_generator import PatientNetworkGenerator, build_knn_graph_with_masks
from hetero_base_models.train_hybridkg import (compute_link_loss, 
                            split_edges,
                            evaluate,
                            evaluate_link,
                            train_one_epoch,
                            train,
                            test,
                            build_x_dict,
                            set_seed
                            )
from hetero_base_models.utilities import (convert_to_hetero_data, 
                                          bridge_names_to_indices,
                                        assign_kg_by_EdgeScore,
                                        get_top_k_assignments,
                                        convert_to_hetero_data, 
                                        bridge_names_to_indices)
from hetero_base_models.base_models import get_model
import argparse
import pickle

## 1. Train Sample_Disease & Sample_Healthy networks to Get Sample Embeddings

In [93]:
config = {
'graph_path': "../datasets/Patient_KGs/G_adni_ADKG_ecdf.pkl",
'output_dir': "../results/GatedMLP",
'model':'gat',
'hidden_channels': 128,
'out_channels': 64,
'num_layers': 3,
'heads': 4,
'dropout': 0.3,
'epochs': 100,
'lr': 1e-3,
'weight_decay': 1e-5,
'lambda_link': 0.5,
'val_ratio': 0.15,
'test_ratio':0.15,
'seed':42,
'device': 'cuda',
'assign_method':'cls',
'confidence': 0.6
}
# convert config to Namespace
args = argparse.Namespace(**config)

In [94]:
def main(args):
    os.makedirs(args.output_dir, exist_ok=True)
    # Setup
    set_seed(args.seed)
    device = torch.device(args.device if torch.cuda.is_available() else "cpu")

    # 1. Prepare HeteroData
    with open(args.graph_path, "rb") as f:
        G = pickle.load(f)
    data, node_mappings = convert_to_hetero_data(G)

    # Build features
    data.x_dict = build_x_dict(data)

    # 2. Edge split
    edge_index_dict = {
        etype: data[etype].edge_index
        for etype in data.edge_types
    }
    train_edges, val_edges, test_edges = split_edges(
        edge_index_dict,
        val_ratio=args.val_ratio,
        test_ratio=args.test_ratio,
        seed=args.seed
    )
    # Labels
    y = data["Patient"].y
    num_classes = int(y.max().item() + 1) if y.dim() == 1 else y.size(-1)
    print(f"Number of classes: {num_classes}")

    # build model
    print("\nStart building model...\n")
    model = get_model(
    data=data,
    model_type=args.model,
    hidden_channels=args.hidden_channels,
    out_channels=args.out_channels,
    num_layers=args.num_layers,
    heads=args.heads,
    dropout=args.dropout,
    num_classes=num_classes,
    device=device
    )

    optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=args.lr,
    weight_decay=args.weight_decay
    )

    # 4. Train: only by link prediction loss
    print("\nStart training...\n")
    print(f"lambda_link = {args.lambda_link}")

    model, train_history = train(
        model=model,
        data=data,
        train_edges=train_edges,
        val_edges=val_edges,
        optimizer=optimizer,
        device=device,
        epochs=args.epochs,
        lambda_link=args.lambda_link
    )

    # get sample embeddings
    model.eval()
    with torch.no_grad():
        x_dict = {k: v.to(device) for k, v in data.x_dict.items()}
        #edge_index_dict = {k: v.to(device) for k, v in data.edge_index_dict.items()}
        edge_index_dict = {
            k: v.to(device) for k, v in edge_index_dict.items()
        }
        # forward: encode ONLY with TRAIN edges
        z_dict = model.encode(x_dict, edge_index_dict)
    return z_dict, data

In [95]:
z_disease, data_disease = main(args)

Starting conversion from NetworkX to HeteroData...
HeteroData created: 16 node types, 610 edge types.
Number of classes: 2

Start building model...


Start training...

lambda_link = 0.5


Training HeteoGNN:   1%|          | 1/100 [00:11<18:40, 11.32s/it]

Epoch 0 | {'loss': 0.7478122711181641, 'cls_loss': 0.6929691433906555, 'link_loss': 0.8026554584503174} | Val {'acc': 0.5, 'f1_macro': 0.3333333333333333, 'auroc': 0.5004329004329005, 'auprc': 0.5824583895582495}


Training HeteoGNN:   2%|▏         | 2/100 [00:23<19:31, 11.96s/it]

Epoch 1 | {'loss': 101.62905883789062, 'cls_loss': 1.6157824993133545, 'link_loss': 201.642333984375} | Val {'acc': 0.5147058823529411, 'f1_macro': 0.5060532687651331, 'auroc': 0.43636363636363634, 'auprc': 0.4947902781205722}


...

Epoch 97 | {'loss': 0.6537693738937378, 'cls_loss': 0.6745167970657349, 'link_loss': 0.633021891117096} | Val {'acc': 0.5441176470588235, 'f1_macro': 0.49605546258666033, 'auroc': 0.4822510822510822, 'auprc': 0.4966033530194874}


Training HeteoGNN:  99%|█████████▉| 99/100 [15:14<00:08,  8.81s/it]

Epoch 98 | {'loss': 0.6391899585723877, 'cls_loss': 0.6607648730278015, 'link_loss': 0.6176151037216187} | Val {'acc': 0.5735294117647058, 'f1_macro': 0.5374149659863945, 'auroc': 0.4848484848484849, 'auprc': 0.49899513113973387}


Training HeteoGNN: 100%|██████████| 100/100 [15:22<00:00,  9.23s/it]

Epoch 99 | {'loss': 0.6400551199913025, 'cls_loss': 0.6686577200889587, 'link_loss': 0.6114525198936462} | Val {'acc': 0.5441176470588235, 'f1_macro': 0.5137254901960784, 'auroc': 0.48744588744588746, 'auprc': 0.4978438897704674}


In [105]:
args.graph_path = "../datasets/Patient_KGs/G_adni_HealthyKG_ecdf.pkl"
z_healthy, data_healthy = main(args)

Starting conversion from NetworkX to HeteroData...
HeteroData created: 23 node types, 508 edge types.
Number of classes: 2

Start building model...


Start training...

lambda_link = 0.5


Training HeteoGNN:   1%|          | 1/100 [00:09<15:07,  9.16s/it]

Epoch 0 | {'loss': 0.6963257789611816, 'cls_loss': 0.6935428380966187, 'link_loss': 0.6991087198257446} | Val {'acc': 0.4852941176470588, 'f1_macro': 0.4188034188034188, 'auroc': 0.5229437229437229, 'auprc': 0.5697124336631153}


Training HeteoGNN:   2%|▏         | 2/100 [00:16<13:34,  8.31s/it]

Epoch 1 | {'loss': 19.354177474975586, 'cls_loss': 0.7133909463882446, 'link_loss': 37.994964599609375} | Val {'acc': 0.5441176470588235, 'f1_macro': 0.44277028813111285, 'auroc': 0.5246753246753246, 'auprc': 0.5533084113766578}


...

Epoch 97 | {'loss': 0.5047944784164429, 'cls_loss': 0.518835186958313, 'link_loss': 0.49075379967689514} | Val {'acc': 0.4852941176470588, 'f1_macro': 0.4657687991021324, 'auroc': 0.4268398268398269, 'auprc': 0.45970582138797983}


Training HeteoGNN:  99%|█████████▉| 99/100 [12:23<00:08,  8.19s/it]

Epoch 98 | {'loss': 0.5088869333267212, 'cls_loss': 0.513317346572876, 'link_loss': 0.5044565796852112} | Val {'acc': 0.5, 'f1_macro': 0.47786811201445345, 'auroc': 0.4268398268398268, 'auprc': 0.45625866083644073}


Training HeteoGNN: 100%|██████████| 100/100 [12:32<00:00,  7.52s/it]

Epoch 99 | {'loss': 0.5026626586914062, 'cls_loss': 0.5082085728645325, 'link_loss': 0.49711674451828003} | Val {'acc': 0.47058823529411764, 'f1_macro': 0.4631578947368421, 'auroc': 0.4233766233766234, 'auprc': 0.4566722814301592}


## 2. Train a Gated MLP

In [106]:
h_healthy = z_healthy['Patient']
h_healthy.size()
h_healthy

tensor([[-0.4989, -0.3539, -0.0580,  ..., -0.1924,  0.7422,  0.4619],
        [ 0.0336,  0.0257, -0.4841,  ..., -0.6766,  0.5754,  0.0800],
        [-0.3185, -0.2539, -0.2631,  ...,  0.3338,  0.4082, -0.5790],
        ...,
        [-0.5892, -0.2313,  0.5319,  ..., -0.2113,  0.6484,  0.2514],
        [-0.1588,  0.1131,  0.1421,  ..., -0.9718,  0.6164,  0.2887],
        [-0.3643, -0.0233,  0.8764,  ...,  0.3340,  1.0490,  0.2813]])

In [96]:
h_disease = z_disease['Patient']
h_disease.size()
h_disease

tensor([[-0.8776,  0.4331, -0.6840,  ..., -2.1480,  3.3953, -1.1695],
        [-0.6788,  0.4999, -0.0558,  ..., -2.1962,  2.9243, -0.7484],
        [-0.9296,  0.3372, -0.3215,  ..., -2.1955,  2.9461, -0.7832],
        ...,
        [-0.8460,  0.9791,  0.2302,  ..., -2.3857,  2.7178, -0.2929],
        [-1.0443,  0.7205, -0.3646,  ..., -2.6813,  3.3580, -1.1475],
        [-0.9228,  1.1644, -0.3568,  ..., -2.1876,  2.6678, -0.7115]])

In [129]:
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)
import torch.nn as nn

class MLPClassifier(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=2, dropout=0.2):
        super().__init__()
        self.dims = [in_channels] + [hidden_channels] * (num_layers - 1) + [out_channels]
        self.layers = torch.nn.ModuleList()
        
        for i in range(len(self.dims) - 1):
            self.layers.append(torch.nn.Linear(self.dims[i], self.dims[i+1]))
            
        self.dropout = dropout

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        return x

class GatedMLP(nn.Module):
    def __init__(self, num_patients, input_dim, hidden_dim, num_classes):
        super(GatedMLP, self).__init__()
        
        self.gate_network = nn.Linear(input_dim * 2, 1) 

        # Final Classification Head
        self.classifier = MLPClassifier(input_dim, hidden_dim, num_classes, num_layers=2, dropout=0.2)

    def forward(self, h_disease, h_healthy):
        #h_disease = F.normalize(h_disease, p=2, dim=1)
        #h_healthy = F.normalize(h_healthy, p=2, dim=1)
    
        # gate should rely on features, otherwise when new samples come in, it won't have alpha
        combined = torch.cat([h_disease, h_healthy], dim=1)
        alpha = torch.sigmoid(self.gate_network(combined))
        self.alpha = alpha
        h_fused = alpha * h_disease + (1 - alpha) * h_healthy
        
        # classify
        logits = self.classifier(h_fused)
        return logits, alpha
        


def train_mlp_step(model, h_disease, h_healthy, labels, train_mask, optimizer, lambda_gate=0.5, lambda_corr=0.1):
    model.train()
    optimizer.zero_grad()

    # 1. Forward pass for the whole graph
    logits, alphas = model(h_disease, h_healthy)

    # 2. Classification Loss (masked)
    cls_loss = F.cross_entropy(logits[train_mask], labels[train_mask])

    # 3. Label-Supervised Gating Loss (masked)
    # Forces alpha[i] to match label[i] for training samples
    gating_loss = F.binary_cross_entropy(alphas[train_mask].squeeze(), labels[train_mask].float())

    # 4. Correlation Regularizer
    # Using only the training nodes to enforce diversity between the two KGs
    # We want the dot product of the two representations to be small (independent)
    h_dis_train = h_disease[train_mask]
    h_hea_train = h_healthy[train_mask]
    
    # L1 norm of the correlation matrix
    corr_matrix = torch.matmul(h_dis_train, h_hea_train.t())
    corr_loss = torch.norm(corr_matrix, p=1) / (h_dis_train.shape[0]**2)

    # 5. Total Loss
    total_loss = cls_loss + (lambda_gate * gating_loss) + (lambda_corr * corr_loss)

    total_loss.backward()
    # Check if the classifier weights are getting gradients
    print(f"Classifier Grad: {model.classifier.layers[0].weight.grad.abs().mean().item()}")

    # Check if the Alpha parameters are getting gradients
    #print(f"Alphas Grad: {model.alpha.grad.abs().mean().item()}")
    
    optimizer.step()

    return {"total_loss":total_loss.item(), 
            "cls_loss":cls_loss.item(), 
            "gating_loss":gating_loss.item()}

def evaluate_mlp(model, labels, h_disease, h_healthy, val_mask):
    model.eval()

    with torch.no_grad():
        logits, alphas = model(h_disease, h_healthy)
    
    logits_val = logits[val_mask]
    y_val = labels[val_mask]
    y_val = y_val.cpu().numpy()

    probs = F.softmax(logits[val_mask], dim=-1).cpu().numpy()
    preds = probs.argmax(axis=1)

    auroc = roc_auc_score(y_val, probs[:, 1])
    auprc = average_precision_score(y_val, probs[:, 1])
       

    results = {
            "Accuracy": float(accuracy_score(y_val, preds)),
            "F1_score": float(f1_score(y_val, preds)),
            "AUROC": float(auroc),
            "AUPRC": float(auprc),
        }

    return results

def train_mlp(model, data, h_disease, h_healthy, optimizer, epochs, lambda_gate=0.5, lambda_corr=0.1):
    best_val = 0
    best_state = None
    train_history = {}
    labels = data['Patient'].y
    train_mask = data['Patient'].train_mask
    val_mask = data['Patient'].val_mask

    for epoch in tqdm(range(epochs), desc="Training GatedMLP"):
        epoch_record = {}

        losses = train_mlp_step(model, 
                                h_disease, 
                                h_healthy, 
                                labels, 
                                train_mask, 
                                optimizer, 
                                lambda_gate=lambda_gate, 
                                lambda_corr=lambda_corr)
        
        val_metrics = evaluate_mlp(model, 
                                   labels, 
                                   h_disease, 
                                   h_healthy, 
                                   val_mask)
        epoch_record['loss'] = losses
        epoch_record['validation'] = val_metrics
        train_history[epoch] = epoch_record

        if val_metrics['Accuracy'] > best_val:
            best_val = val_metrics['Accuracy']
            best_state = model.state_dict()
        
        print(f"Epoch {epoch} | Losses {losses} | Val {val_metrics}")
    
    model.load_state_dict(best_state)
    
    return model, train_history

In [135]:
# main() script for training mlp
mlp = GatedMLP(num_patients=h_healthy.size(0), 
               input_dim=h_healthy.size(1), 
               hidden_dim=128, 
               num_classes=2)
               
optimizer = torch.optim.AdamW(
    mlp.parameters(),
    lr=0.01,
    weight_decay=args.weight_decay
    )

In [136]:
mlp, train_history = train_mlp(model=mlp, 
                               data=data_disease, 
                               h_disease=h_disease, 
                               h_healthy=h_healthy, 
                               optimizer=optimizer, 
                               epochs=100, 
                               lambda_gate=1.0, 
                               lambda_corr=0.0)

Training GatedMLP:   3%|▎         | 3/100 [00:00<00:04, 23.55it/s]

Classifier Grad: 0.0005748009425587952
Epoch 0 | Losses {'total_loss': 1.3871150016784668, 'cls_loss': 0.6950858235359192, 'gating_loss': 0.6920291781425476} | Val {'Accuracy': 0.6176470588235294, 'F1_score': 0.7045454545454546, 'AUROC': 0.6069264069264069, 'AUPRC': 0.6140854948504982}
Classifier Grad: 0.0004337162827141583
Epoch 1 | Losses {'total_loss': 1.3469854593276978, 'cls_loss': 0.6621338129043579, 'gating_loss': 0.6848516464233398} | Val {'Accuracy': 0.6029411764705882, 'F1_score': 0.7032967032967034, 'AUROC': 0.6112554112554113, 'AUPRC': 0.5808574616319628}
Classifier Grad: 0.0012605213560163975
Epoch 2 | Losses {'total_loss': 1.3162403106689453, 'cls_loss': 0.6515085697174072, 'gating_loss': 0.6647318005561829} | Val {'Accuracy': 0.5, 'F1_score': 0.2608695652173913, 'AUROC': 0.6285714285714286, 'AUPRC': 0.6171092568835612}
Classifier Grad: 0.0027504749596118927
Epoch 3 | Losses {'total_loss': 1.3254423141479492, 'cls_loss': 0.6786363124847412, 'gating_loss': 0.64680594205856

Training GatedMLP:  19%|█▉        | 19/100 [00:00<00:01, 61.71it/s]

Epoch 8 | Losses {'total_loss': 1.2473640441894531, 'cls_loss': 0.6257514357566833, 'gating_loss': 0.6216126680374146} | Val {'Accuracy': 0.5735294117647058, 'F1_score': 0.6233766233766234, 'AUROC': 0.5852813852813853, 'AUPRC': 0.5336141659654778}
Classifier Grad: 0.0007758297142572701
Epoch 9 | Losses {'total_loss': 1.240859031677246, 'cls_loss': 0.6154882907867432, 'gating_loss': 0.6253708004951477} | Val {'Accuracy': 0.6323529411764706, 'F1_score': 0.6987951807228916, 'AUROC': 0.580952380952381, 'AUPRC': 0.5292380242223432}
Classifier Grad: 0.000531058176420629
Epoch 10 | Losses {'total_loss': 1.2288691997528076, 'cls_loss': 0.6031621098518372, 'gating_loss': 0.6257070899009705} | Val {'Accuracy': 0.6470588235294118, 'F1_score': 0.7142857142857143, 'AUROC': 0.5783549783549784, 'AUPRC': 0.5255937071813211}
Classifier Grad: 0.0008772979490458965
Epoch 11 | Losses {'total_loss': 1.2168688774108887, 'cls_loss': 0.5969113111495972, 'gating_loss': 0.6199575066566467} | Val {'Accuracy': 0.

Training GatedMLP:  36%|███▌      | 36/100 [00:00<00:00, 73.66it/s]

...

Epoch 63 | Losses {'total_loss': 0.942764163017273, 'cls_loss': 0.36938783526420593, 'gating_loss': 0.5733763575553894} | Val {'Accuracy': 0.5294117647058824, 'F1_score': 0.5897435897435898, 'AUROC': 0.5722943722943723, 'AUPRC': 0.5531051765171536}
Classifier Grad: 0.0010509317507967353
Epoch 64 | Losses {'total_loss': 0.9169369339942932, 'cls_loss': 0.34459972381591797, 'gating_loss': 0.5723372101783752} | Val {'Accuracy': 0.5882352941176471, 'F1_score': 0.6585365853658537, 'AUROC': 0.5792207792207792, 'AUPRC': 0.5545590845988186}
Classifier Grad: 0.0011749609839171171
Epoch 65 | Losses {'total_loss': 0.9185872077941895, 'cls_loss': 0.3466399908065796, 'gating_loss': 0.5719472169876099} | Val {'Accuracy': 0.5735294117647058, 'F1_score': 0.6233766233766234, 'AUROC': 0.5731601731601732, 'AUPRC': 0.5575302157770298}
Classifier Grad: 0.0009728238801471889
Epoch 66 | Losses {'total_loss': 0.9228911399841309, 'cls_loss': 0.3507838547229767, 'gating_loss': 0.5721072554588318} | Val {'Accurac

Training GatedMLP:  92%|█████████▏| 92/100 [00:01<00:00, 68.54it/s]

Classifier Grad: 0.0009028324857354164
Epoch 79 | Losses {'total_loss': 0.8286783695220947, 'cls_loss': 0.2611752450466156, 'gating_loss': 0.5675031542778015} | Val {'Accuracy': 0.5441176470588235, 'F1_score': 0.5974025974025974, 'AUROC': 0.5593073593073593, 'AUPRC': 0.5550145472747093}
Classifier Grad: 0.001101961825042963
Epoch 80 | Losses {'total_loss': 0.8217360973358154, 'cls_loss': 0.25447264313697815, 'gating_loss': 0.5672634243965149} | Val {'Accuracy': 0.5588235294117647, 'F1_score': 0.6153846153846154, 'AUROC': 0.5575757575757576, 'AUPRC': 0.5569923911930719}
Classifier Grad: 0.0010735619580373168
Epoch 81 | Losses {'total_loss': 0.8228995203971863, 'cls_loss': 0.2558109164237976, 'gating_loss': 0.5670886039733887} | Val {'Accuracy': 0.5588235294117647, 'F1_score': 0.5945945945945946, 'AUROC': 0.5645021645021645, 'AUPRC': 0.5568616319844683}
Classifier Grad: 0.0008315164013765752
Epoch 82 | Losses {'total_loss': 0.8236470222473145, 'cls_loss': 0.2557120621204376, 'gating_loss

Training GatedMLP: 100%|██████████| 100/100 [00:01<00:00, 65.43it/s]

Epoch 94 | Losses {'total_loss': 0.7753331661224365, 'cls_loss': 0.21015046536922455, 'gating_loss': 0.5651826858520508} | Val {'Accuracy': 0.6029411764705882, 'F1_score': 0.64, 'AUROC': 0.5627705627705628, 'AUPRC': 0.5634717671540639}
Classifier Grad: 0.0013168565928936005
Epoch 95 | Losses {'total_loss': 0.7696360349655151, 'cls_loss': 0.2036920040845871, 'gating_loss': 0.5659440159797668} | Val {'Accuracy': 0.5588235294117647, 'F1_score': 0.5833333333333334, 'AUROC': 0.5645021645021644, 'AUPRC': 0.5646201231034293}
Classifier Grad: 0.0018154132412746549
Epoch 96 | Losses {'total_loss': 0.791347861289978, 'cls_loss': 0.22527243196964264, 'gating_loss': 0.5660754442214966} | Val {'Accuracy': 0.5441176470588235, 'F1_score': 0.5753424657534246, 'AUROC': 0.5662337662337662, 'AUPRC': 0.568486319816544}
Classifier Grad: 0.0010282796574756503
Epoch 97 | Losses {'total_loss': 0.7599844932556152, 'cls_loss': 0.19565418362617493, 'gating_loss': 0.5643302798271179} | Val {'Accuracy': 0.55882352

In [139]:
# test on test data
test_metrics = evaluate_mlp(mlp, 
                            data_disease['Patient'].y, 
                            h_disease, 
                            h_healthy, 
                            data_disease['Patient'].test_mask)
test_metrics

{'Accuracy': 0.5942028985507246,
 'F1_score': 0.65,
 'AUROC': 0.6907563025210084,
 'AUPRC': 0.7483688431509515}